In [ ]:
# --- Deep learning framework ---
from transformers import AutoModel, AutoConfig
import torch
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torchvision import transforms

# --- Data processing ---
import h5py
import os
import numpy as np
import pandas as pd
import json

# --- Image processing ---
from PIL import ImageFile, Image

# --- Spatial transcriptomics ---
import scanpy as sc

In [ ]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

## 1. Load TITAN Model

In [ ]:
# TITAN encodes patch features + spatial coords into a slide-level global embedding
titan = AutoModel.from_pretrained(
    "./TITAN",
    trust_remote_code=True,
    local_files_only=True
)
titan = titan.to(device)

## 2. Verify Extracted Features

In [ ]:
feature_dict = {i: get_features(feaure_dir, i) for i in names}
for name in names:
    print(f"{name} features shape: {feature_dict[name].shape}")

## 3. TITAN Slide Encoding

In [ ]:
# Prepare example data from sample A1
feature_ori = feature_dict['A1']
centers = (np.floor(exp_dict['A1'].obs[['pixel_x', 'pixel_y']]).values).astype(int)
coords = torch.from_numpy(centers)
feature_ori = feature_ori.unsqueeze(0)   # [1, N, 1536]
coords = coords.unsqueeze(0)             # [1, N, 2]
patch_size_lv0 = 1

In [ ]:
# Load reference TITAN H5 file for comparison
demo_h5_path = "./TITAN/TCGA-PC-A5DK-01Z-00-DX1.C2D3BC09-411F-46CF-811B-FDBA7C2A295B.h5"
file = h5py.File(demo_h5_path, 'r')
features = torch.from_numpy(file['features'][:])          # [1, 3190, 768]
coords_ref = torch.from_numpy(file['coords'][:])          # [1, 3190, 2]
patch_size_lv0_ref = file['coords'].attrs['patch_size_level0']  # 1024

print(f"Reference features: {features.shape}")
print(f"Reference coords: {coords_ref.shape}")
print(f"Patch size at level 0: {patch_size_lv0_ref}")

In [ ]:
# Encode slide embedding with mixed precision
with torch.autocast('cuda', torch.float16), torch.inference_mode():
    features = features.to(device)
    coords_ref = coords_ref.to(device)
    slide_embedding = titan.encode_slide_from_patch_features(
        features, coords_ref, patch_size_lv0_ref
    )

print(f"Slide embedding shape: {slide_embedding.shape}")  # [3190, 768]